# Base File Aggregation — North Region

Combines every salesperson's `Base_File_Updation` workbook into one consolidated
`combined_output` file — with live formulas preserved, every original column
kept (nothing dropped), and column-total formulas in row 1, matching the
agreed output layout.

**What this notebook does**
1. Reads every input `.xlsx` file in `input_files/` (auto-decrypting any that turn out to be password protected).
2. For each of the three data sheets (`INR Sheet`, `USD Sheet`, `Direct Sheet`), stacks every file's data rows together — **by column position against the fixed layout below, not by matching each file's own header text.** This matters: if one file's header label gets accidentally overwritten (it has happened before), that file's data still lands in the right place, because only the fixed layout's column order is trusted for meaning, never an individual file's own header row.
3. Keeps every original column, in its original order — nothing is dropped, nothing is reordered.
4. Recreates every formula-driven column (`Value = Qty x Price`, and the roll-up totals) as a **live formula**, re-pointed at the correct row.
5. Adds a **row 1 column-total** (`=SUBTOTAL(9, ...)`) for every column that had one in the original template, extended to cover the full combined range.
6. Recalculates every formula and confirms there are zero formula errors.
7. Writes the combined workbook to `output_files/`, timestamped, plus a **Validation Report** sheet documenting row counts and any notes worth a human's attention.

**To run this next month:** drop the new month's files into `input_files/` and run all cells. No reference/template file is needed — the layout is fixed below.


In [15]:
import os
import io
import re
import glob
import shutil
import tempfile
import subprocess
from datetime import datetime

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

try:
    import msoffcrypto
except ImportError:
    msoffcrypto = None


## 1. Configuration

In [16]:
# --- folders ---------------------------------------------------------------
# --- folders ---------------------------------------------------------------
from pathlib import Path
INPUT_DIR = Path("input_files/received_files")     # every salesperson's source workbook lives here
OUTPUT_DIR = "output_files"   # the combined file is written here

# --- password used ONLY if a workbook turns out to be encrypted ----------
INPUT_PASSWORD = "raja"

# --- output file name: combined_output_<YYYYMMDD>_<HHMMSS>.xlsx -----------
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"combined_output_{RUN_TIMESTAMP}.xlsx")

# --- path to the recalc helper that ships with the xlsx skill -------------
RECALC_SCRIPT = "/mnt/skills/public/xlsx/scripts/recalc.py"

# --- discover input files automatically ------------------------------------
INPUT_FILES = sorted(glob.glob(os.path.join(INPUT_DIR, "*.xlsx")))

# --- sheet names as they appear (with the small whitespace inconsistencies
#     found in the real files) mapped to their canonical name -------------
SHEET_ALIASES = {
    "INR Sheet": ["INR Sheet", "INR Sheet "],
    "USD Sheet": ["USD Sheet", "USD Sheet "],
    "Direct Sheet": ["Direct Sheet", "Direct Sheet "],
}
CANONICAL_SHEETS = ["INR Sheet", "USD Sheet", "Direct Sheet"]

HEADER_ROW = 2       # row containing column names in every SOURCE workbook
DATA_START_ROW = 3   # first row of actual data in every SOURCE workbook

# --- output styling --------------------------------------------------------
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_ALIGN = Alignment(horizontal="center", vertical="center")
DATA_FONT = Font(name="Calibri", size=11)

print(f"Found {len(INPUT_FILES)} input file(s) in '{INPUT_DIR}':")
for f in INPUT_FILES:
    print(" -", os.path.basename(f))


Found 7 input file(s) in 'input_files/received_files':
 - Abhishekh-Base File Updation North Region -Oct-26 working file.xlsx
 - Arpit-Base File Updation North Region -Oct-26 working file.xlsx
 - Copy of Harsh-Base File Updation North Region -Oct-26 working file.xlsx
 - Dhirender-Base File Updation North Region -Oct-26.xlsx
 - Kuldeep-Base File Updation North Region -Oct-26 working file.xlsx
 - Mahesh-Base File Updation North Region -Oct-26 working file.xlsx
 - Suraj-Base File Updation North Region -Oct-26 working file.xlsx


## 2. Fixed output layout

Column order, widths, and number formats for each sheet — captured once
from the agreed template, so every run produces an identically-shaped file
regardless of which salesperson files are fed in, and **every original
column is kept** (nothing is dropped this time, even columns that are
usually empty). Update this block only if the agreed layout itself
changes.

In [17]:
OUTPUT_SCHEMA = {
    "INR Sheet": {
        "headers": ['Currency', 'Zone', 'Region', 'Customer Code', 'Customer Name', 'Updated sales', 'Sales CSR', 'Principal', 'Make', 'PM Names', 'PM MSO', 'Part Number', 'Existing/New', 'EP Prices', 'Sep Month Total Inventory qty', 'All Month Budget Qty In K Value', 'All Month Budget (B) Values In K', 'Sep-26 to Mar-27 (F) Value', 'Sep-26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr(B) Value', 'Apr(F) Qty', 'Apr(F) Value', 'Apr Incoming', "May'26(B) Qty", 'May(B) Value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July B value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug(F) Value', 'Aug Incoming', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep(F) Qty', 'Sep(F) Value', 'Sep Incoming', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct(F) Qty', 'Oct(F) Value', 'Oct Incoming', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov(F) Qty', 'Nov(F) Value', 'Nov Incoming', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27 (B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming', 'Remarks'],
        "widths": {'A': 12.6328125, 'B': 8.453125, 'C': 14.7265625, 'D': 19.0, 'E': 31.453125, 'F': 15.0, 'G': 17.54296875, 'H': 11.453125, 'I': 11.453125, 'J': 14.81640625, 'K': 14.26953125, 'L': 19.453125, 'M': 16.26953125, 'N': 13.90625, 'O': 20.0, 'P': 24.54296875, 'Q': 24.08984375, 'R': 23.6328125, 'S': 20.6328125, 'T': 17.26953125, 'U': 16.453125, 'V': 14.08984375, 'W': 16.26953125, 'X': 17.453125, 'Y': 17.90625, 'Z': 17.54296875, 'AA': 15.1796875, 'AB': 17.453125, 'AC': 20.36328125, 'AD': 18.36328125, 'AE': 16.90625, 'AF': 14.54296875, 'AG': 16.81640625, 'AH': 19.6328125, 'AI': 17.6328125, 'AJ': 15.7265625, 'AK': 15.0, 'AL': 17.1796875, 'AM': 20.08984375, 'AN': 17.54296875, 'AO': 17.1796875, 'AP': 14.453125, 'AQ': 16.6328125, 'AR': 17.7265625, 'AS': 17.6328125, 'AT': 17.26953125, 'AU': 14.54296875, 'AV': 16.81640625, 'AW': 17.81640625, 'AX': 17.453125, 'AY': 17.08984375, 'AZ': 14.26953125, 'BA': 16.54296875, 'BB': 17.6328125, 'BC': 17.7265625, 'BD': 17.36328125, 'BE': 14.54296875, 'BF': 16.81640625, 'BG': 17.81640625, 'BH': 17.7265625, 'BI': 17.453125, 'BJ': 14.6328125, 'BK': 16.90625, 'BL': 17.90625, 'BM': 17.08984375, 'BN': 16.81640625, 'BO': 14.0, 'BP': 16.08984375, 'BQ': 20.453125, 'BR': 17.54296875, 'BS': 17.1796875, 'BT': 14.453125, 'BU': 16.6328125, 'BV': 21.36328125, 'BW': 18.54296875, 'BX': 17.26953125, 'BY': 14.453125, 'BZ': 16.6328125, 'CA': 20.90625, 'CB': 62.36328125, 'CC': 33.6328125},
        "number_formats": {'Currency': 'General', 'Zone': 'General', 'Region': 'General', 'Customer Code': 'General', 'Customer Name': 'General', 'Updated sales': 'General', 'Sales CSR': 'General', 'Principal': 'General', 'Make': 'General', 'PM Names': 'General', 'PM MSO': 'General', 'Part Number': 'General', 'Existing/New': 'General', 'EP Prices': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', 'Sep Month Total Inventory qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget Qty In K Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget (B) Values In K': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 to Mar-27 (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 to Mar-27 Incoming Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Apr'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "May'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May(B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "June'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "July'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July B value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Aug'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Sep'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Oct'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Nov'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Dec'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Jan'27(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Feb'27(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Mar'27 (B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Remarks': 'General'},
    },
    "USD Sheet": {
        "headers": ['Currency', 'Zone', 'Region', 'Customer Code', 'Customer Name', 'Segment', 'Sub Segment', 'End Customer Name', 'End Customer Segment', 'End Customer Sub-Segment', 'Updates Sale person Name', 'Sales CSR', 'Principal', 'Make', 'PM Names', 'PM CSR', 'Part Number', 'Existing/New', 'EP in US $', 'Sep-26 Month Total Inventory Qty', 'All Month Budget Qty In K', 'All Month Budget(B) Value in K', 'Sep-26 to Mar-27 (F) Value', 'Sep-26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr(B) Value', 'Apr(F) Qty', 'Apr(F) Value', 'Apr Incoming', "May'26(B) Qty", 'May(B) Value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July B value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug(F) Value', 'Aug Incoming', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep(F) Qty', 'Sep(F) Value', 'Sep Incoming', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct(F) Qty', 'Oct(F) Value', 'Oct Incoming', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov(F) Qty', 'Nov(F) Value', 'Nov Incoming', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27 (B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming', 'Remarks', None, None],
        "widths": {'A': 13.81640625, 'B': 9.26953125, 'C': 16.08984375, 'D': 22.81640625, 'E': 41.36328125, 'F': 16.08984375, 'G': 24.7265625, 'H': 65.0, 'I': 28.26953125, 'J': 33.08984375, 'K': 17.0, 'L': 14.36328125, 'M': 14.7265625, 'N': 16.36328125, 'O': 19.08984375, 'P': 15.0, 'Q': 17.26953125, 'R': 18.6328125, 'S': 15.453125, 'T': 24.6328125, 'U': 22.26953125, 'V': 25.453125, 'W': 25.54296875, 'X': 24.7265625, 'Y': 19.26953125, 'Z': 18.1796875, 'AA': 15.81640625, 'AB': 17.90625, 'AC': 18.90625, 'AD': 19.81640625, 'AE': 19.1796875, 'AF': 16.7265625, 'AG': 18.81640625, 'AH': 22.1796875, 'AI': 20.1796875, 'AJ': 18.26953125, 'AK': 15.90625, 'AL': 18.0, 'AM': 21.36328125, 'AN': 19.54296875, 'AO': 17.0, 'AP': 16.453125, 'AQ': 18.54296875, 'AR': 21.90625, 'AS': 19.54296875, 'AT': 18.81640625, 'AU': 16.08984375, 'AV': 18.1796875, 'AW': 19.26953125, 'AX': 19.26953125, 'AY': 18.54296875, 'AZ': 15.81640625, 'BA': 17.90625, 'BB': 18.90625, 'BC': 19.1796875, 'BD': 18.36328125, 'BE': 15.6328125, 'BF': 17.7265625, 'BG': 18.81640625, 'BH': 19.54296875, 'BI': 18.81640625, 'BJ': 16.08984375, 'BK': 18.1796875, 'BL': 19.26953125, 'BM': 19.453125, 'BN': 18.6328125, 'BO': 15.90625, 'BP': 18.0, 'BQ': 19.1796875, 'BR': 18.81640625, 'BS': 18.1796875, 'BT': 15.453125, 'BU': 17.453125, 'BV': 22.0, 'BW': 19.1796875, 'BX': 18.36328125, 'BY': 15.6328125, 'BZ': 17.7265625, 'CA': 22.7265625, 'CB': 20.36328125, 'CC': 18.81640625, 'CD': 16.08984375, 'CE': 18.1796875, 'CF': 22.7265625, 'CG': 119.7265625, 'CH': 18.7265625},
        "number_formats": {'Currency': 'General', 'Zone': 'General', 'Region': 'General', 'Customer Code': 'General', 'Customer Name': 'General', 'Segment': 'General', 'Sub Segment': 'General', 'End Customer Name': 'General', 'End Customer Segment': 'General', 'End Customer Sub-Segment': 'General', 'Updates Sale person Name': 'General', 'Sales CSR': 'General', 'Principal': 'General', 'Make': 'General', 'PM Names': 'General', 'PM CSR': 'General', 'Part Number': 'General', 'Existing/New': 'General', 'EP in US $': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 Month Total Inventory Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget Qty In K': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget(B) Value in K': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 to Mar-27 (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 to Mar-27 Incoming Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Apr'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "May'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May(B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "June'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "July'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July B value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Aug'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Sep'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Oct'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Nov'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Dec'26(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Jan'27(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Feb'27(B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Mar'27 (B) Qty": '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar(F) Qty': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Remarks': 'General'},
    },
    "Direct Sheet": {
        "headers": ['Currency', 'ZONE', 'Region', 'Customer Code', 'Customer Name', 'Updated sales', 'Sales CSR', 'Principal', 'Make', 'PM Names', 'PM CSR', 'Part Number', 'Existing/New', 'EP in US $', 'Sep-26 Month Inventory', 'All Month Budget Qty In K Value', 'All Month Budget (B) Values In K', 'Sep-26 to Mar-27 (F) Value', 'Sep 26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr B value', 'Apr (F) Qty', 'Apr(F) Value', 'Apr (Incoming)', "May'26(B) Qty", 'May (B) value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July (B) Value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug (F) Value', 'Aug (Incoming)', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep (F) Qty', 'Sep (F) Value', 'Sep (Incoming)', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct (F) Qty', 'Oct (F) Value', 'Oct (Incoming)', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov (F) Qty', 'Nov (F) Value', 'Nov (Incoming)', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27(B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming', 'Remarks'],
        "widths": {'A': 14.6328125, 'B': 10.90625, 'C': 15.453125, 'D': 20.36328125, 'E': 43.6328125, 'F': 24.7265625, 'G': 17.90625, 'H': 21.81640625, 'I': 21.81640625, 'J': 14.90625, 'K': 12.1796875, 'L': 16.6328125, 'M': 18.6328125, 'N': 18.1796875, 'O': 21.90625, 'P': 19.08984375, 'Q': 19.6328125, 'R': 21.7265625, 'S': 23.08984375, 'T': 21.54296875, 'U': 19.54296875, 'V': 18.81640625, 'W': 20.90625, 'X': 24.36328125, 'Y': 22.0, 'Z': 21.36328125, 'AA': 19.26953125, 'AB': 21.81640625, 'AC': 24.54296875, 'AD': 22.7265625, 'AE': 21.0, 'AF': 18.6328125, 'AG': 20.7265625, 'AH': 24.26953125, 'AI': 21.81640625, 'AJ': 21.36328125, 'AK': 19.08984375, 'AL': 21.08984375, 'AM': 24.6328125, 'AN': 21.90625, 'AO': 21.54296875, 'AP': 18.81640625, 'AQ': 21.26953125, 'AR': 24.36328125, 'AS': 21.81640625, 'AT': 21.36328125, 'AU': 19.08984375, 'AV': 21.08984375, 'AW': 24.6328125, 'AX': 21.6328125, 'AY': 21.26953125, 'AZ': 18.90625, 'BA': 21.0, 'BB': 24.08984375, 'BC': 21.90625, 'BD': 21.54296875, 'BE': 19.1796875, 'BF': 21.26953125, 'BG': 24.7265625, 'BH': 21.90625, 'BI': 21.54296875, 'BJ': 18.81640625, 'BK': 20.90625, 'BL': 22.453125, 'BM': 21.26953125, 'BN': 20.90625, 'BO': 18.0, 'BP': 20.1796875, 'BQ': 24.90625, 'BR': 21.6328125, 'BS': 21.26953125, 'BT': 18.36328125, 'BU': 20.6328125, 'BV': 25.6328125, 'BW': 21.81640625, 'BX': 21.36328125, 'BY': 18.6328125, 'BZ': 20.7265625, 'CA': 25.453125, 'CB': 77.6328125, 'CC': 8.7265625},
        "number_formats": {'Currency': 'General', 'ZONE': 'General', 'Region': 'General', 'Customer Code': 'General', 'Customer Name': 'General', 'Updated sales': 'General', 'Sales CSR': 'General', 'Principal': 'General', 'Make': 'General', 'PM Names': 'General', 'PM CSR': 'General', 'Part Number': 'General', 'Existing/New': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'EP in US $': '_ * #,##0.000_ ;_ * \\-#,##0.000_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 Month Inventory': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget Qty In K Value': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', 'All Month Budget (B) Values In K': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', 'Sep-26 to Mar-27 (F) Value': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', 'Sep 26 to Mar-27 Incoming Qty': '_ * #,##0.00_ ;_ * \\-#,##0.00_ ;_ * "-"??_ ;_ @_ ', "Apr'26(B) Qty": 'General', 'Apr B value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr (F) Qty': 'General', 'Apr(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Apr (Incoming)': 'General', "May'26(B) Qty": 'General', 'May (B) value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (F) Qty': 'General', 'May (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'May (Incoming)': 'General', "June'26(B) Qty": 'General', 'Jun (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (F) Qty': 'General', 'Jun (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jun (Incoming)': 'General', "July'26(B) Qty": 'General', 'July (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (F) Qty': 'General', 'July (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'July (Incoming)': 'General', "Aug'26(B) Qty": 'General', 'Aug (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug(F) Qty': 'General', 'Aug (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Aug (Incoming)': 'General', "Sep'26(B) Qty": 'General', 'Sep (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep (F) Qty': 'General', 'Sep (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Sep (Incoming)': 'General', "Oct'26(B) Qty": 'General', 'Oct (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct (F) Qty': 'General', 'Oct (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Oct (Incoming)': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Nov'26(B) Qty": 'General', 'Nov (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov (F) Qty': 'General', 'Nov (F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Nov (Incoming)': 'General', "Dec'26(B) Qty": 'General', 'Dec (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec(F) Qty': 'General', 'Dec(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Dec Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Jan'27(B) Qty": 'General', 'Jan (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan(F) Qty': 'General', 'Jan(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Jan-27 Incoming': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', "Feb'27(B) Qty": 'General', 'Feb (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb(F) Qty': 'General', 'Feb(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Feb-27 Incoming': 'General', "Mar'27(B) Qty": 'General', 'Mar (B) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar(F) Qty': 'General', 'Mar(F) Value': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ ', 'Mar-27 Incoming': 'General', 'Remarks': '_ * #,##0_ ;_ * \\-#,##0_ ;_ * "-"??_ ;_ @_ '},
    },
}

## 3. Row 1 column totals

The original per-person templates carry a `=SUBTOTAL(9, ...)` column-total
formula in row 1 for most of the numeric columns (everything from the first
summary column onward). This list — which columns get one — was captured
once from the template, the same way the layout above was, and is just
re-pointed to cover the full combined row range at write time.

In [18]:
ROW1_SUBTOTAL_HEADERS = {
    "INR Sheet": ['Sep Month Total Inventory qty', 'All Month Budget Qty In K Value', 'All Month Budget (B) Values In K', 'Sep-26 to Mar-27 (F) Value', 'Sep-26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr(B) Value', 'Apr(F) Qty', 'Apr(F) Value', 'Apr Incoming', "May'26(B) Qty", 'May(B) Value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July B value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug(F) Value', 'Aug Incoming', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep(F) Qty', 'Sep(F) Value', 'Sep Incoming', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct(F) Qty', 'Oct(F) Value', 'Oct Incoming', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov(F) Qty', 'Nov(F) Value', 'Nov Incoming', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27 (B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming'],
    "USD Sheet": ['Sep-26 Month Total Inventory Qty', 'All Month Budget Qty In K', 'All Month Budget(B) Value in K', 'Sep-26 to Mar-27 (F) Value', 'Sep-26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr(B) Value', 'Apr(F) Qty', 'Apr(F) Value', 'Apr Incoming', "May'26(B) Qty", 'May(B) Value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July B value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug(F) Value', 'Aug Incoming', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep(F) Qty', 'Sep(F) Value', 'Sep Incoming', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct(F) Qty', 'Oct(F) Value', 'Oct Incoming', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov(F) Qty', 'Nov(F) Value', 'Nov Incoming', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27 (B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming'],
    "Direct Sheet": ['All Month Budget Qty In K Value', 'All Month Budget (B) Values In K', 'Sep-26 to Mar-27 (F) Value', 'Sep 26 to Mar-27 Incoming Qty', "Apr'26(B) Qty", 'Apr B value', 'Apr (F) Qty', 'Apr(F) Value', 'Apr (Incoming)', "May'26(B) Qty", 'May (B) value', 'May (F) Qty', 'May (F) Value', 'May (Incoming)', "June'26(B) Qty", 'Jun (B) Value', 'Jun (F) Qty', 'Jun (F) Value', 'Jun (Incoming)', "July'26(B) Qty", 'July (B) Value', 'July (F) Qty', 'July (F) Value', 'July (Incoming)', "Aug'26(B) Qty", 'Aug (B) Value', 'Aug(F) Qty', 'Aug (F) Value', 'Aug (Incoming)', "Sep'26(B) Qty", 'Sep (B) Value', 'Sep (F) Qty', 'Sep (F) Value', 'Sep (Incoming)', "Oct'26(B) Qty", 'Oct (B) Value', 'Oct (F) Qty', 'Oct (F) Value', 'Oct (Incoming)', "Nov'26(B) Qty", 'Nov (B) Value', 'Nov (F) Qty', 'Nov (F) Value', 'Nov (Incoming)', "Dec'26(B) Qty", 'Dec (B) Value', 'Dec(F) Qty', 'Dec(F) Value', 'Dec Incoming', "Jan'27(B) Qty", 'Jan (B) Value', 'Jan(F) Qty', 'Jan(F) Value', 'Jan-27 Incoming', "Feb'27(B) Qty", 'Feb (B) Value', 'Feb(F) Qty', 'Feb(F) Value', 'Feb-27 Incoming', "Mar'27(B) Qty", 'Mar (B) Value', 'Mar(F) Qty', 'Mar(F) Value', 'Mar-27 Incoming'],
}

## 4. Formula templates

The value columns in the source files (e.g. `Apr(B) Value`) and the monthly
roll-up totals (e.g. `All Month Budget Qty In K Value`) are formulas that
reference *other columns in the same row* — e.g. `Value = Qty × EP Price`.

Each template is written in terms of **header names**, not fixed cell
references — at write time, each `{Header Name}` placeholder is resolved to
whichever column that header ends up in, and the row number is filled in.

In [19]:
FORMULA_TEMPLATES = {
    "INR Sheet": {
        'All Month Budget Qty In K Value': "=+({Apr'26(B) Qty}+{May'26(B) Qty}+{June'26(B) Qty}+{July'26(B) Qty}+{Aug'26(B) Qty}+{Sep'26(B) Qty}+{Oct'26(B) Qty}+{Nov'26(B) Qty}+{Dec'26(B) Qty}+{Jan'27(B) Qty}+{Feb'27(B) Qty}+{Mar'27 (B) Qty})/1000",
        'All Month Budget (B) Values In K': '=+({Apr(B) Value}+{May(B) Value}+{Jun (B) Value}+{July B value}+{Aug (B) Value}+{Sep (B) Value}+{Oct (B) Value}+{Nov (B) Value}+{Dec (B) Value}+{Jan (B) Value}+{Feb (B) Value}+{Mar (B) Value})/1000',
        'Sep-26 to Mar-27 (F) Value': '=+(+{Sep(F) Value}+{Oct(F) Value}+{Nov(F) Value}+{Dec(F) Value}+{Jan(F) Value}+{Feb(F) Value}+{Mar(F) Value})',
        'Sep-26 to Mar-27 Incoming Qty': '=+(+{Sep Incoming}+{Oct Incoming}+{Nov Incoming}+{Dec Incoming}+{Jan-27 Incoming}+{Feb-27 Incoming}+{Mar-27 Incoming})',
        'Apr(B) Value': "={Apr'26(B) Qty}*{EP Prices}",
        'Apr(F) Value': '={Apr(F) Qty}*{EP Prices}',
        'May(B) Value': "={May'26(B) Qty}*{EP Prices}",
        'May (F) Value': '={May (F) Qty}*{EP Prices}',
        'Jun (B) Value': "={June'26(B) Qty}*{EP Prices}",
        'Jun (F) Value': '={Jun (F) Qty}*{EP Prices}',
        'July B value': "={July'26(B) Qty}*{EP Prices}",
        'July (F) Value': '={July (F) Qty}*{EP Prices}',
        'Aug (B) Value': "={Aug'26(B) Qty}*{EP Prices}",
        'Aug(F) Value': '={Aug(F) Qty}*{EP Prices}',
        'Sep (B) Value': "={Sep'26(B) Qty}*{EP Prices}",
        'Sep(F) Value': '={Sep(F) Qty}*{EP Prices}',
        'Oct (B) Value': "={Oct'26(B) Qty}*{EP Prices}",
        'Oct(F) Value': '={Oct(F) Qty}*{EP Prices}',
        'Nov (B) Value': "={Nov'26(B) Qty}*{EP Prices}",
        'Nov(F) Value': '={Nov(F) Qty}*{EP Prices}',
        'Dec (B) Value': "={Dec'26(B) Qty}*{EP Prices}",
        'Dec(F) Value': '={Dec(F) Qty}*{EP Prices}',
        'Jan (B) Value': "={Jan'27(B) Qty}*{EP Prices}",
        'Jan(F) Value': '={Jan(F) Qty}*{EP Prices}',
        'Feb (B) Value': "={Feb'27(B) Qty}*{EP Prices}",
        'Feb(F) Value': '={Feb(F) Qty}*{EP Prices}',
        'Mar (B) Value': "={Mar'27 (B) Qty}*{EP Prices}",
        'Mar(F) Value': '={Mar(F) Qty}*{EP Prices}',
    },
    "USD Sheet": {
        'All Month Budget Qty In K': "=+({Apr'26(B) Qty}+{May'26(B) Qty}+{June'26(B) Qty}+{July'26(B) Qty}+{Aug'26(B) Qty}+{Sep'26(B) Qty}+{Oct'26(B) Qty}+{Nov'26(B) Qty}+{Dec'26(B) Qty}+{Jan'27(B) Qty}+{Feb'27(B) Qty}+{Mar'27 (B) Qty})/1000",
        'All Month Budget(B) Value in K': '=+({Apr(B) Value}+{May(B) Value}+{Jun (B) Value}+{July B value}+{Aug (B) Value}+{Sep (B) Value}+{Oct (B) Value}+{Nov (B) Value}+{Dec (B) Value}+{Jan (B) Value}+{Feb (B) Value}+{Mar (B) Value})/1000',
        'Sep-26 to Mar-27 (F) Value': '=+(+{Sep(F) Value}+{Oct(F) Value}+{Nov(F) Value}+{Dec(F) Value}+{Jan(F) Value}+{Feb(F) Value}+{Mar(F) Value})',
        'Sep-26 to Mar-27 Incoming Qty': '=+(+{Sep Incoming}+{Oct Incoming}+{Nov Incoming}+{Dec Incoming}+{Jan-27 Incoming}+{Feb-27 Incoming}+{Mar-27 Incoming})',
        'Apr(B) Value': "={Apr'26(B) Qty}*{EP in US $}",
        'Apr(F) Value': '={Apr(F) Qty}*{EP in US $}',
        'May(B) Value': "={May'26(B) Qty}*{EP in US $}",
        'May (F) Value': '={May (F) Qty}*{EP in US $}',
        'Jun (B) Value': "={June'26(B) Qty}*{EP in US $}",
        'Jun (F) Value': '={Jun (F) Qty}*{EP in US $}',
        'July B value': "={July'26(B) Qty}*{EP in US $}",
        'July (F) Value': '={July (F) Qty}*{EP in US $}',
        'Aug (B) Value': "={Aug'26(B) Qty}*{EP in US $}",
        'Aug(F) Value': '={Aug(F) Qty}*{EP in US $}',
        'Sep (B) Value': "={Sep'26(B) Qty}*{EP in US $}",
        'Sep(F) Value': '={Sep(F) Qty}*{EP in US $}',
        'Oct (B) Value': "={Oct'26(B) Qty}*{EP in US $}",
        'Oct(F) Value': '={Oct(F) Qty}*{EP in US $}',
        'Nov (B) Value': "={Nov'26(B) Qty}*{EP in US $}",
        'Nov(F) Value': '={Nov(F) Qty}*{EP in US $}',
        'Dec (B) Value': "={Dec'26(B) Qty}*{EP in US $}",
        'Dec(F) Value': '={Dec(F) Qty}*{EP in US $}',
        'Jan (B) Value': "={Jan'27(B) Qty}*{EP in US $}",
        'Jan(F) Value': '={Jan(F) Qty}*{EP in US $}',
        'Feb (B) Value': "={Feb'27(B) Qty}*{EP in US $}",
        'Feb(F) Value': '={Feb(F) Qty}*{EP in US $}',
        'Mar (B) Value': "={Mar'27 (B) Qty}*{EP in US $}",
        'Mar(F) Value': '={Mar(F) Qty}*{EP in US $}',
    },
    "Direct Sheet": {
        'All Month Budget Qty In K Value': "=+({Apr'26(B) Qty}+{May'26(B) Qty}+{June'26(B) Qty}+{July'26(B) Qty}+{Aug'26(B) Qty}+{Sep'26(B) Qty}+{Oct'26(B) Qty}+{Nov'26(B) Qty}+{Dec'26(B) Qty}+{Jan'27(B) Qty}+{Feb'27(B) Qty}+{Mar'27(B) Qty})/1000",
        'All Month Budget (B) Values In K': '=+({Apr B value}+{May (B) value}+{Jun (B) Value}+{July (B) Value}+{Aug (B) Value}+{Sep (B) Value}+{Oct (B) Value}+{Nov (B) Value}+{Dec (B) Value}+{Jan (B) Value}+{Feb (B) Value}+{Mar (B) Value})/1000',
        'Sep-26 to Mar-27 (F) Value': '=+(+{Sep (F) Value}+{Oct (F) Value}+{Nov (F) Value}+{Dec(F) Value}+{Jan(F) Value}+{Feb(F) Value}+{Mar(F) Value})',
        'Sep 26 to Mar-27 Incoming Qty': '=+{Sep (Incoming)}+{Oct (Incoming)}+{Nov (Incoming)}+{Dec Incoming}+{Jan-27 Incoming}+{Feb-27 Incoming}+{Mar-27 Incoming}',
        'Apr B value': "={Apr'26(B) Qty}*{EP in US $}",
        'Apr(F) Value': '={Apr (F) Qty}*{EP in US $}',
        'May (B) value': "={May'26(B) Qty}*{EP in US $}",
        'May (F) Value': '={May (F) Qty}*{EP in US $}',
        'Jun (B) Value': "={June'26(B) Qty}*{EP in US $}",
        'Jun (F) Value': '={Jun (F) Qty}*{EP in US $}',
        'July (B) Value': "={July'26(B) Qty}*{EP in US $}",
        'July (F) Value': '={July (F) Qty}*{EP in US $}',
        'Aug (B) Value': "={Aug'26(B) Qty}*{EP in US $}",
        'Aug (F) Value': '={Aug(F) Qty}*{EP in US $}',
        'Sep (B) Value': "={Sep'26(B) Qty}*{EP in US $}",
        'Sep (F) Value': '={Sep (F) Qty}*{EP in US $}',
        'Oct (B) Value': "={Oct'26(B) Qty}*{EP in US $}",
        'Oct (F) Value': '={Oct (F) Qty}*{EP in US $}',
        'Nov (B) Value': "={Nov'26(B) Qty}*{EP in US $}",
        'Nov (F) Value': '={Nov (F) Qty}*{EP in US $}',
        'Dec (B) Value': "={Dec'26(B) Qty}*{EP in US $}",
        'Dec(F) Value': '={Dec(F) Qty}*{EP in US $}',
        'Jan (B) Value': "={Jan'27(B) Qty}*{EP in US $}",
        'Jan(F) Value': '={Jan(F) Qty}*{EP in US $}',
        'Feb (B) Value': "={Feb'27(B) Qty}*{EP in US $}",
        'Feb(F) Value': '={Feb(F) Qty}*{EP in US $}',
        'Mar (B) Value': "={Mar'27(B) Qty}*{EP in US $}",
        'Mar(F) Value': '={Mar(F) Qty}*{EP in US $}',
    },
}

In [20]:
PLACEHOLDER_RE = re.compile(r"\{([^{}]+)\}")


def referenced_headers(template):
    """Every {Header Name} placeholder a formula template depends on."""
    return PLACEHOLDER_RE.findall(template)


## 5. Helper functions — reading & decrypting source files

In [21]:
def person_name(filepath):
    """The salesperson's name is the first token before '-' in the filename,
    with a leading 'Copy of' stripped off (some return files get renamed that
    way by whoever forwards them)."""
    return os.path.basename(filepath).split("-")[0].replace("Copy of", "").strip()


def clean_header(value):
    """Normalise a header string: collapse newlines/repeated spaces, strip ends."""
    if value is None:
        return None
    text = str(value).replace("\n", " ")
    return " ".join(text.split())


def open_workbook(path, password=INPUT_PASSWORD, data_only=False):
    """
    Open a workbook, transparently decrypting it first if msoffcrypto reports
    it as password protected. Falls back to a normal open otherwise.
    """
    is_encrypted = False
    if msoffcrypto is not None:
        try:
            with open(path, "rb") as fh:
                is_encrypted = msoffcrypto.OfficeFile(fh).is_encrypted()
        except Exception:
            is_encrypted = False

    if not is_encrypted:
        return openpyxl.load_workbook(path, data_only=data_only)

    decrypted = io.BytesIO()
    with open(path, "rb") as fh:
        office_file = msoffcrypto.OfficeFile(fh)
        office_file.load_key(password=password)
        office_file.decrypt(decrypted)
    decrypted.seek(0)
    return openpyxl.load_workbook(decrypted, data_only=data_only)


def find_sheet(wb, canonical_name):
    """Return the actual sheet name in this workbook matching a canonical name, or None."""
    for alias in SHEET_ALIASES[canonical_name]:
        if alias in wb.sheetnames:
            return alias
    return None


def recalc_with_libreoffice(path):
    """Safety net: only triggered if a formula's cached value is missing."""
    tmp_copy = os.path.join(tempfile.mkdtemp(), os.path.basename(path))
    shutil.copy(path, tmp_copy)
    subprocess.run(
        ["python3", RECALC_SCRIPT, tmp_copy, "60"],
        check=False, capture_output=True, text=True,
    )
    return tmp_copy


## 6. Reading a sheet — by column position, not by header name

**This is the key design decision in this notebook.** Earlier versions
matched columns across files by header *text*. That works fine until one
file's header row gets corrupted — which has happened in practice (a
column label accidentally overwritten with a part number). If columns are
matched by name, a corrupted label makes that file's data vanish from the
correct column and reappear as a bogus new one.

Instead, every file's data is read by **column position** against the fixed
layout in `OUTPUT_SCHEMA` above. Column position doesn't change even when a
label does — as long as the columns themselves haven't moved (that's a
structural problem the separate validator checks for beforehand), reading
by position is immune to a corrupted header label.

A quick, informational-only check still compares each file's own header
text against the expected layout at each position, and notes any mismatch
in the Validation Report sheet — but it never blocks the run, since the
data itself is still read correctly either way.

In [22]:
VALIDATION_LOG = []   # each entry: {"File", "Sheet", "Issue", "Details"}


def read_sheet_by_position(path, canonical_name, n_cols):
    """
    Reads one sheet's data rows purely by column position (1..n_cols),
    matching the fixed OUTPUT_SCHEMA layout — never by looking up columns
    via this file's own header text. Returns a list of rows, each row a
    list of (value, is_formula) tuples, or None if the file has no such sheet.
    """
    wb_formula = open_workbook(path, data_only=False)
    actual_name = find_sheet(wb_formula, canonical_name)
    if actual_name is None:
        return None

    ws_formula = wb_formula[actual_name]
    wb_values = open_workbook(path, data_only=True)
    ws_values = wb_values[actual_name]

    # detect formulas whose cached value is missing -> needs a recalc pass
    needs_recalc = False
    for row in ws_formula.iter_rows(min_row=DATA_START_ROW, max_col=n_cols):
        for cell in row:
            if isinstance(cell.value, str) and cell.value.startswith("="):
                if ws_values.cell(row=cell.row, column=cell.column).value is None:
                    needs_recalc = True
                    break
        if needs_recalc:
            break
    if needs_recalc:
        recalced_path = recalc_with_libreoffice(path)
        wb_values = open_workbook(recalced_path, data_only=True)
        ws_values = wb_values[actual_name]

    # informational-only header check (never blocks the run)
    own_headers = [clean_header(ws_formula.cell(row=HEADER_ROW, column=c).value) for c in range(1, n_cols + 1)]
    schema_headers = OUTPUT_SCHEMA[canonical_name]["headers"]
    person = person_name(path)
    for i, (own, expected) in enumerate(zip(own_headers, schema_headers), start=1):
        if own != expected:
            VALIDATION_LOG.append({
                "File": person, "Sheet": canonical_name,
                "Issue": "INFO: this file's header text differs from the standard layout at this column "
                         "(data was still read correctly by position)",
                "Details": f"column {get_column_letter(i)}: file says {own!r}, expected {expected!r}",
            })

    rows = []
    for r in range(DATA_START_ROW, ws_values.max_row + 1):
        row_vals = []
        any_val = False
        for c in range(1, n_cols + 1):
            fv = ws_formula.cell(row=r, column=c).value
            is_formula = isinstance(fv, str) and fv.startswith("=")
            v = ws_values.cell(row=r, column=c).value
            if v is not None:
                any_val = True
            row_vals.append((v, is_formula))
        if not any_val:
            continue
        rows.append(row_vals)
    return rows


## 7. Aggregation logic

In [23]:
def aggregate_sheet(canonical_name, files):
    """Stack every file's rows for one sheet, in alphabetical order by salesperson."""
    schema_headers = OUTPUT_SCHEMA[canonical_name]["headers"]
    n_cols = len(schema_headers)
    ordered_files = sorted(files, key=person_name)

    combined_rows = []   # each row: list of (value, is_formula), aligned to schema_headers by position
    row_counts = {}
    for f in ordered_files:
        rows = read_sheet_by_position(f, canonical_name, n_cols)
        name = person_name(f)
        if rows is None:
            row_counts[name] = 0
            continue
        row_counts[name] = len(rows)
        combined_rows.extend(rows)

    for name, count in row_counts.items():
        VALIDATION_LOG.append({
            "File": name, "Sheet": canonical_name,
            "Issue": "INFO: rows contributed",
            "Details": str(count),
        })

    return {"row_counts": row_counts, "combined_rows": combined_rows}


def usable_formula_templates(canonical_name, final_order):
    """
    Formula templates for this sheet where every header they depend on
    exists in the fixed layout. (With the fixed layout always keeping every
    original column, this should be all of them — this check is a safety
    net in case the layout is ever trimmed in the future.)
    """
    templates = FORMULA_TEMPLATES.get(canonical_name, {})
    usable = {}
    for h, tmpl in templates.items():
        if h not in final_order:
            continue
        missing_refs = [r for r in referenced_headers(tmpl) if r not in final_order]
        if missing_refs:
            VALIDATION_LOG.append({
                "File": "-", "Sheet": canonical_name,
                "Issue": "INFO: formula not recreated (a column it depends on isn't in the fixed layout)",
                "Details": f"{h} needs {missing_refs}",
            })
        else:
            usable[h] = tmpl
    return usable


def scan_formula_errors(rows, headers):
    """Flag any Excel error strings (#REF!, #DIV/0!, etc.) that made it into cached values."""
    error_markers = ("#REF!", "#DIV/0!", "#VALUE!", "#NAME?", "#N/A", "#NULL!", "#NUM!")
    bad = []
    for i, row_vals in enumerate(rows, start=3):
        for h, (v, _is_f) in zip(headers, row_vals):
            if isinstance(v, str) and v.strip() in error_markers:
                bad.append((i, h, v))
    return bad


## 8. Run the aggregation for every sheet

In [24]:
files_by_sheet = {c: [] for c in CANONICAL_SHEETS}
for f in INPUT_FILES:
    wb = open_workbook(f)
    for c in CANONICAL_SHEETS:
        if find_sheet(wb, c):
            files_by_sheet[c].append(f)

SHEET_RESULTS = {}
for canonical in CANONICAL_SHEETS:
    agg = aggregate_sheet(canonical, files_by_sheet[canonical])
    final_order = OUTPUT_SCHEMA[canonical]["headers"]

    errors = scan_formula_errors(agg["combined_rows"], final_order)
    if errors:
        for row_num, h, v in errors[:20]:
            VALIDATION_LOG.append({
                "File": "-", "Sheet": canonical,
                "Issue": "ERROR: formula error in source data",
                "Details": f"row {row_num}, column '{h}': {v}",
            })

    usable_templates = usable_formula_templates(canonical, final_order)
    SHEET_RESULTS[canonical] = {
        "agg": agg, "final_order": final_order, "usable_templates": usable_templates,
    }
    print(f"{canonical}: {len(agg['combined_rows'])} rows x {len(final_order)} columns  "
          f"({len(usable_templates)} formula columns)  "
          f"(per-file counts: {agg['row_counts']})")


INR Sheet: 1048 rows x 80 columns  (28 formula columns)  (per-file counts: {'Abhishekh': 73, 'Arpit': 150, 'Dhirender': 12, 'Harsh': 254, 'Kuldeep': 259, 'Mahesh': 157, 'Suraj': 143})
USD Sheet: 1109 rows x 87 columns  (28 formula columns)  (per-file counts: {'Abhishekh': 26, 'Arpit': 209, 'Harsh': 79, 'Kuldeep': 335, 'Mahesh': 156, 'Suraj': 304})
Direct Sheet: 58 rows x 80 columns  (28 formula columns)  (per-file counts: {'Harsh': 7, 'Kuldeep': 12, 'Mahesh': 7, 'Suraj': 32})


## 9. Write the combined workbook

For every cell: if its header has a usable formula template *and* the
source cell for this row was itself a formula, write a live formula
(header placeholders resolved to this run's actual column letters + row
number). Otherwise, write the plain value.

In [25]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

wb_out = openpyxl.Workbook()
wb_out.remove(wb_out.active)

formula_cell_counts = {}

for canonical in CANONICAL_SHEETS:
    result = SHEET_RESULTS[canonical]
    final_order = result["final_order"]
    combined_rows = result["agg"]["combined_rows"]
    usable_templates = result["usable_templates"]

    header_to_letter = {h: get_column_letter(i + 1) for i, h in enumerate(final_order) if h}

    # keep the trailing-space sheet name used by the source templates
    actual_sheet_name = SHEET_ALIASES[canonical][1]
    ws = wb_out.create_sheet(actual_sheet_name)
    widths = OUTPUT_SCHEMA[canonical].get("widths", {})
    number_formats = OUTPUT_SCHEMA[canonical].get("number_formats", {})

    for c_idx, h in enumerate(final_order, start=1):
        if h is None:
            continue
        cell = ws.cell(row=HEADER_ROW, column=c_idx, value=h)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = HEADER_ALIGN
        col_letter = get_column_letter(c_idx)
        if col_letter in widths:
            ws.column_dimensions[col_letter].width = widths[col_letter]

    formula_cells, static_cells = 0, 0
    last_row = HEADER_ROW + len(combined_rows)
    for r_idx, row_vals in enumerate(combined_rows, start=DATA_START_ROW):
        for c_idx, h in enumerate(final_order, start=1):
            if h is None:
                continue
            value, is_formula_source = row_vals[c_idx - 1]

            if h in usable_templates and is_formula_source:
                tmpl = usable_templates[h]

                def resolve(m, _r=r_idx):
                    return f"{header_to_letter[m.group(1)]}{_r}"

                formula = PLACEHOLDER_RE.sub(resolve, tmpl)
                cell = ws.cell(row=r_idx, column=c_idx, value=formula)
                formula_cells += 1
            else:
                cell = ws.cell(row=r_idx, column=c_idx, value=value)
                static_cells += 1

            cell.font = DATA_FONT
            nf = number_formats.get(h)
            if nf and nf != "General":
                cell.number_format = nf

    # row 1: column-total formulas, extended to cover the full combined range
    for h in ROW1_SUBTOTAL_HEADERS.get(canonical, []):
        if h in header_to_letter:
            c_idx = final_order.index(h) + 1
            letter = header_to_letter[h]
            ws.cell(row=1, column=c_idx, value=f"=SUBTOTAL(9,{letter}{DATA_START_ROW}:{letter}{last_row})")

    ws.freeze_panes = f"A{DATA_START_ROW}"
    formula_cell_counts[canonical] = (formula_cells, static_cells)
    print(f"{canonical}: {formula_cells} live formula cells, {static_cells} static value cells")

print("\nSheets written:", wb_out.sheetnames)


INR Sheet: 29338 live formula cells, 54502 static value cells
USD Sheet: 29908 live formula cells, 64357 static value cells
Direct Sheet: 1624 live formula cells, 3016 static value cells

Sheets written: ['INR Sheet ', 'USD Sheet ', 'Direct Sheet ']


## 10. Validation Report sheet

In [26]:
ws = wb_out.create_sheet("Validation Report")
report_headers = ["File", "Sheet", "Issue", "Details"]
for c_idx, h in enumerate(report_headers, start=1):
    cell = ws.cell(row=1, column=c_idx, value=h)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
    cell.alignment = HEADER_ALIGN

error_rows = [e for e in VALIDATION_LOG if e["Issue"].startswith("ERROR")]
info_rows = [e for e in VALIDATION_LOG if not e["Issue"].startswith("ERROR")]

if not error_rows:
    ws.cell(row=2, column=1, value="-")
    ws.cell(row=2, column=2, value="-")
    ws.cell(row=2, column=3, value="No issues found")
    ws.cell(row=2, column=4, value=None)
    next_row = 3
else:
    next_row = 2
    for e in error_rows:
        ws.cell(row=next_row, column=1, value=e["File"])
        ws.cell(row=next_row, column=2, value=e["Sheet"])
        ws.cell(row=next_row, column=3, value=e["Issue"])
        ws.cell(row=next_row, column=4, value=e["Details"])
        next_row += 1

for e in info_rows:
    ws.cell(row=next_row, column=1, value=e["File"])
    ws.cell(row=next_row, column=2, value=e["Sheet"])
    ws.cell(row=next_row, column=3, value=e["Issue"])
    ws.cell(row=next_row, column=4, value=e["Details"])
    next_row += 1

for col, width in zip("ABCD", [16, 14, 55, 60]):
    ws.column_dimensions[col].width = width
ws.freeze_panes = "A2"

print(f"Validation Report: {len(error_rows)} error row(s), {len(info_rows)} info row(s)")


Validation Report: 0 error row(s), 81 info row(s)


## 11. Save the workbook

Saved with `fullCalcOnLoad` set, so Excel recalculates every formula
automatically the moment the file is opened — no separate calculation
engine is required on the machine running this notebook. If LibreOffice
happens to be available, it's used for an extra QA pass that confirms zero
formula errors; otherwise that check is skipped without affecting the
output.

In [27]:
wb_out.calculation.fullCalcOnLoad = True
wb_out.save(OUTPUT_FILE)
print("Saved:", OUTPUT_FILE)

recalc_script_available = os.path.exists(RECALC_SCRIPT) and shutil.which("soffice") is not None

recalc_status = {"ran": False, "total_formulas": None, "total_errors": None, "note": ""}

if recalc_script_available:
    recalc_result = subprocess.run(
        ["python3", RECALC_SCRIPT, OUTPUT_FILE, "180"],
        capture_output=True, text=True,
    )
    print(recalc_result.stdout)
    if recalc_result.returncode == 0:
        try:
            import json as _json
            parsed = _json.loads(recalc_result.stdout)
            recalc_status.update(
                ran=True,
                total_formulas=parsed.get("total_formulas"),
                total_errors=parsed.get("total_errors"),
                note="LibreOffice QA recalculation completed.",
            )
        except Exception:
            recalc_status["note"] = "LibreOffice ran but its output could not be parsed."
    else:
        recalc_status["note"] = "LibreOffice recalc check FAILED to run -- see stderr below."
        print("Recalc check did not complete -- see stderr:")
        print(recalc_result.stderr)
else:
    recalc_status["note"] = (
        "LibreOffice recalc check skipped (not available on this machine). "
        "Not a problem -- Excel will recalculate every formula automatically "
        "when the file is opened, because fullCalcOnLoad is set."
    )
    print(recalc_status["note"])


Saved: output_files/combined_output_20260927_154954.xlsx
LibreOffice recalc check skipped (not available on this machine). Not a problem -- Excel will recalculate every formula automatically when the file is opened, because fullCalcOnLoad is set.


## 12. Run summary — read this first

In [28]:
print("=" * 70)
print("BASE FILE AGGREGATION -- RUN SUMMARY")
print("=" * 70)
print(f"Run finished : {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Input files  : {len(INPUT_FILES)} file(s) from '{INPUT_DIR}'")
print("-" * 70)

print(f"{'Sheet':<14}{'Rows':>8}{'Columns':>10}{'Formulas':>12}{'Static':>10}")
for canonical in CANONICAL_SHEETS:
    result = SHEET_RESULTS[canonical]
    n_rows = len(result["agg"]["combined_rows"])
    n_cols = len([h for h in result["final_order"] if h])
    f_cells, s_cells = formula_cell_counts.get(canonical, (0, 0))
    print(f"{canonical:<14}{n_rows:>8}{n_cols:>10}{f_cells:>12}{s_cells:>10}")
print("-" * 70)

print(f"Validation Report : {len(error_rows)} error(s), {len(info_rows)} info note(s)")
if error_rows:
    print("\n  ERRORS FOUND:")
    for e in error_rows:
        print(f"   - [{e['Sheet']}] {e['Issue']} | {e['Details']}")
else:
    print("  No errors.")

header_notes = [e for e in info_rows if "header text differs" in e["Issue"]]
if header_notes:
    print(f"\n  {len(header_notes)} file(s) have a header label that doesn't match the standard layout")
    print("  (their data was still read correctly, by column position) -- see 'Validation Report' for details.")

print("-" * 70)
print("Formula recalculation:")
print("  Excel        : will auto-recalculate every formula on open (fullCalcOnLoad set)")
if recalc_status["ran"]:
    print(f"  LibreOffice QA check : {recalc_status['total_errors']} error(s) across "
          f"{recalc_status['total_formulas']} formulas")
else:
    print(f"  LibreOffice QA check : skipped -- {recalc_status['note']}")
print("=" * 70)

recalc_failed = recalc_status["ran"] and (recalc_status["total_errors"] or 0) > 0
overall_ok = (len(error_rows) == 0) and not recalc_failed

if overall_ok:
    print("RESULT: ALL GOOD -- the output file is ready to share.")
else:
    print("RESULT: NEEDS REVIEW -- see the errors above and the 'Validation Report'")
    print("        sheet inside the output file before sharing it.")
print("=" * 70)


BASE FILE AGGREGATION -- RUN SUMMARY
Run finished : 27-Sep-2026 15:50:05
Output file  : output_files/combined_output_20260927_154954.xlsx
Input files  : 7 file(s) from 'input_files/received_files'
----------------------------------------------------------------------
Sheet             Rows   Columns    Formulas    Static
INR Sheet         1048        80       29338     54502
USD Sheet         1109        85       29908     64357
Direct Sheet        58        80        1624      3016
----------------------------------------------------------------------
Validation Report : 0 error(s), 81 info note(s)
  No errors.

  64 file(s) have a header label that doesn't match the standard layout
  (their data was still read correctly, by column position) -- see 'Validation Report' for details.
----------------------------------------------------------------------
Formula recalculation:
  Excel        : will auto-recalculate every formula on open (fullCalcOnLoad set)
  LibreOffice QA check : skippe